## 2. Install pinned dependencies

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import os

candidate_roots = [
    Path.cwd(),
    Path('/content/drive/MyDrive/University/8th Semester/BDA/CEP/EpiScore'),
    Path('/content/drive/MyDrive/University/8th Semester/BDA/CEP/EpiScope'),
    Path('/content/drive/MyDrive/University/8th Semester/BDA/CEP/episcope'),
    Path('/content/drive/MyDrive/University/8th Semester/BDA/episcope'),
]
project_root = next((p.resolve() for p in candidate_roots if (p / 'scripts' / 'run_pipeline.py').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Set candidate_roots to the Drive folder containing scripts/run_pipeline.py.')
os.chdir(project_root)
print(f'EpiScope project: {project_root}')


In [4]:
ls

AI_USAGE.md  data/               episcore.ipynb  README.md         scripts/
analytics/   docker-compose.yml  generator/      requirements.txt  serving/
dashboard/   docs/               pipeline/       results/          tests/


In [9]:
!pip install -q -r requirements.txt

In [11]:
import sys
import os
import getpass

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

os.environ['EPISCOPE_HASH_SALT'] = getpass.getpass('EPISCOPE_HASH_SALT: ')
print(f'Python: {sys.executable}')


EPISCOPE_HASH_SALT: ··········
Python: /usr/bin/python3


## 3. Verify Spark and PuLP

In [14]:
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/check_spark.py'], check=True)
subprocess.run([sys.executable, '-c', 'import pyspark, pulp; print(pyspark.__version__, pulp.__version__)'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-c', 'import pyspark, pulp; print(pyspark.__version__, pulp.__version__)'], returncode=0)

## 4. Smoke run

This validates generator, Bronze, Silver, Gold, analytics, serving, and dashboard export with 10,000 records.

In [20]:
subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '10000', '--seed', '42'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--rows', '10000', '--seed', '42'], returncode=0)

## 5. Assignment-scale run

The specification requires at least 25,000,000 encounter records at SF=1.

In [21]:
subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '25000000', '--seed', '42'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--rows', '25000000', '--seed', '42'], returncode=0)

## 6. Scaling experiment

In [28]:
# Running scale: 6,250,000 rows
import subprocess
import sys
subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '6250000', '--seed', '42'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--rows', '6250000', '--seed', '42'], returncode=0)

In [31]:
# Running scale: 12,500,000 rows (with auto-CRC cleanup to avoid Google Drive FUSE sync issue)
import subprocess
import sys
from pathlib import Path

# Remove conflicting CRC checksum files to prevent EOFException
print("Cleaning up CRC checksum files in 'data' to prevent Google Drive sync errors...")
for crc_file in Path('data').rglob('.*.crc'):
    try:
        crc_file.unlink()
    except Exception:
        pass

subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '12500000', '--seed', '42'], check=True)

Cleaning up CRC checksum files in 'data' to prevent Google Drive sync errors...


CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--rows', '12500000', '--seed', '42'], returncode=0)

In [35]:
# Running scale: 25,000,000 rows
import subprocess
import sys
from pathlib import Path

def clean_crc():
    print("Cleaning up CRC checksum files in 'data' to prevent Google Drive sync errors...")
    for crc_file in Path('data').rglob('.*.crc'):
        try:
            crc_file.unlink()
        except Exception:
            pass

clean_crc()

subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '25000000', '--seed', '42'], check=True)

Cleaning up CRC checksum files in 'data' to prevent Google Drive sync errors...


CompletedProcess(args=['/usr/bin/python3', 'scripts/run_pipeline.py', '--rows', '25000000', '--seed', '42'], returncode=0)

## 7. Batch repeatability smoke run

This repeats the same small batch configuration. The separate inventory section below demonstrates key-based incremental replay protection.


In [ ]:
for _ in range(2):
    subprocess.run([sys.executable, 'scripts/run_pipeline.py', '--rows', '10000', '--seed', '42'], check=True)

## 8. One-day incremental inventory feed

The assignment describes daily facility stock and bed-occupancy data as an incremental source. This section generates one synthetic snapshot and runs it through its own Bronze, Silver, and Gold paths. It does not edit `pipeline/bronze.py` or replace the encounter CSV.


In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([
    sys.executable, 'generator/generate_inventory_drop.py',
    '--snapshot-date', '2025-12-31', '--seed', '999'
], check=True)

# Clear only stale Spark CRC sidecars for this feed before a rerun.
inventory_paths = [Path('data/bronze/inventory_snapshots'), Path('data/silver/inventory_snapshots'), Path('data/silver/inventory_quarantine'), Path('data/gold/daily_inventory_by_district')]
for inventory_path in inventory_paths:
    if inventory_path.exists():
        for crc_file in inventory_path.rglob('.*.crc'):
            crc_file.unlink(missing_ok=True)


Run the incremental pipeline twice. On a fresh `data/bronze/inventory_snapshots/` path, the first pass should append one row per facility. The replay should append zero rows for the same `(snapshot_date, facility_id)` keys. Save the actual output from your Colab session as evidence.


In [ ]:
subprocess.run([sys.executable, 'pipeline/inventory.py'], check=True)

# Google Drive can retain stale Spark CRC sidecars after a write. Remove only
# inventory-feed CRC files before replaying the same Parquet path.
from pathlib import Path
for inventory_path in [Path('data/bronze/inventory_snapshots'), Path('data/silver/inventory_snapshots'), Path('data/silver/inventory_quarantine'), Path('data/gold/daily_inventory_by_district')]:
    if inventory_path.exists():
        for crc_file in inventory_path.rglob('.*.crc'):
            crc_file.unlink(missing_ok=True)

# Replay the same file to demonstrate idempotency: expected appended rows = 0.
subprocess.run([sys.executable, 'pipeline/inventory.py'], check=True)


In [38]:
# !python scripts/run_pipeline.py --help

usage: run_pipeline.py [-h] [--rows ROWS] [--seed SEED]

Run the EpiScope pipeline.

options:
  -h, --help   show this help message and exit
  --rows ROWS  Number of synthetic encounter rows to generate.
  --seed SEED


## 9. Tests

In [47]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

## 10. Dashboard payload check

In [48]:
dashboard_files = sorted(Path('dashboard/data').glob('*.json'))
total_bytes = sum(path.stat().st_size for path in dashboard_files)
for path in dashboard_files:
    print(f'{path}: {path.stat().st_size / 1024 / 1024:.2f} MB')
print(f'Total: {total_bytes / 1024 / 1024:.2f} MB')
assert total_bytes <= 10 * 1024 * 1024, 'VIS-4 failed: payload exceeds 10 MB'

dashboard/data/alerts.json: 0.00 MB
dashboard/data/calendar.json: 1.13 MB
dashboard/data/cases.json: 0.74 MB
dashboard/data/forecasts.json: 1.01 MB
dashboard/data/resources.json: 0.01 MB
Total: 2.90 MB


## 11. Serve the dashboard

In [49]:
# Start the server in a background process.
dashboard_process = subprocess.Popen([sys.executable, '-m', 'http.server', '8000', '--directory', 'dashboard'])
print('Dashboard server started on port 8000')

Dashboard server started on port 8000


In [52]:
# Colab-only port forwarding.
from google.colab import output
output.serve_kernel_port_as_window(8000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>